In [1]:
import os
import warnings
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip install transformers
!pip install sentencepiece
!pip install datasets
warnings.filterwarnings(
    "ignore", 
    message="Was asked to gather along dimension 0, but all input tensors were scalars"
)

## Local Inference on GPU 
Model page: https://huggingface.co/vesteinn/ScandiBERT

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/vesteinn/ScandiBERT)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("kaggle-scandibert")

import torch
import math
import re
from huggingface_hub import hf_hub_download
from transformers import XLMRobertaTokenizer, AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset, concatenate_datasets
from huggingface_hub import snapshot_download

model_name = "vesteinn/ScandiBERT" # Considering no-faroese variant for slight tokenization difference
labels = ["Norwegian", "Swedish", "Danish", "Non-Scandinavian"]
id2label = {i: l for i, l in enumerate(labels)}
label2id = {l: i for i, l in enumerate(labels)}

print("Downloading model components (ignoring broken tokenizer.json)...")
model_path = snapshot_download(
    repo_id="vesteinn/ScandiBERT", 
    ignore_patterns=["tokenizer.json"]
)

tokenizer = XLMRobertaTokenizer.from_pretrained(model_path)

SENTENCES_PER_CLASS = 3000

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels = len(labels),
    id2label = id2label,  
    label2id = label2id,
    ignore_mismatched_sizes = True,
    problem_type = "multi_label_classification"
)

def get_opus_sentences(config, lang_code, num_sentences=SENTENCES_PER_CLASS):
    print(f"Downloading {lang_code} from {config}...")
    ds = load_dataset("Helsinki-NLP/opus-100", config, split="train")
    ds = ds.filter(lambda sentence: is_dataset_clean(sentence["translation"][lang_code]))
    ds = ds.shuffle(seed=42).select(range(num_sentences))
    return [item[lang_code] for item in ds["translation"]]

def is_dataset_clean(text, instance_threshold=2, min_length=3, max_length=20):
    bad_char_regex = r"[@#$%^&*_=+\[\]{};':\"<>/`~\\|]"
    bad_format_regex = r'[A-Z][a-z]+[A-Z]|%\s*\d|https?://'
    if not isinstance(text, str):
        return False
    if len(text.strip()) < max_length or len(text.split()) < min_length:
        return False
    if len(re.findall(bad_char_regex, text)) >= instance_threshold:
        return False
    if re.search(bad_format_regex, text):
        return False
    return True        

def sent_based_int(percent, sent_const=SENTENCES_PER_CLASS): # Get an integer that's a % of SENTENCES_PER_CLASS
    return int(math.ceil(sent_const * percent))

sample_norwegian_all = get_opus_sentences("en-nb", "nb", sent_based_int(0.75)) + get_opus_sentences("en-nn", "nn", sent_based_int(0.25))
sample_swedish_all = get_opus_sentences("en-sv", "sv")
sample_danish_all = get_opus_sentences("da-en", "da")
nonscandi_configs = [("en-es", "en"), ("en-es", "es"), ("de-en", "de"), ("en-nl", "nl"), ("af-en", "af")]
sample_nonscandi_all = []
for config, code in nonscandi_configs:
    sample_nonscandi_all.extend(get_opus_sentences(config, code, sent_based_int(1 / len(nonscandi_configs))))
print(f"Norwegian corpora collected: {len(sample_norwegian_all)} sentences.")
print(f"Swedish corpora collected: {len(sample_swedish_all)} sentences.")
print(f"Danish corpora collected: {len(sample_danish_all)} sentences.")
print(f"Non-Scandinavian corpora collected: {len(sample_nonscandi_all)} sentences, with {len(sample_nonscandi_all) / len(nonscandi_configs)} sentences per language.")

test_text = "Dette er en norsk setning."
inputs = tokenizer(test_text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
print("\nZero-Shot Classification Probabilities: ")
probs = torch.sigmoid(outputs.logits)
for label, prob in zip(labels, probs[0]): # True "zero-shot", random classification test
    print(f"{label}: {prob.item():.4f}")

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: vesteinn/ScandiBERT
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


README.md: 0.00B [00:00, ?B/s]

en-nb/test-00000-of-00001.parquet:   0%|          | 0.00/223k [00:00<?, ?B/s]

en-nb/train-00000-of-00001.parquet:   0%|          | 0.00/10.2M [00:00<?, ?B/s]

... (52 outputs left out)

Generating train split:   0%|          | 0/275512 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/275512 [00:00<?, ? examples/s]

Norwegian corpora collected: 3000 sentences.
Swedish corpora collected: 3000 sentences.
Danish corpora collected: 3000 sentences.
Non-Scandinavian corpora collected: 3000 sentences, with 600.0 sentences per language.

Zero-Shot Classification Probabilities: 
Norwegian: 0.5289
Swedish: 0.5711
Danish: 0.5239
Non-Scandinavian: 0.5393


In [6]:
from datasets import Dataset, concatenate_datasets, Sequence, Value, Features
from sklearn.model_selection import train_test_split

def smoothed_values(target_index, list_len, SMOOTH_VAL=0.1):
    if target_index > list_len:
        return [None]
    new_list = [None] * list_len
    for i in range(list_len):
        if (i == target_index):
            new_list[i] = 1 - SMOOTH_VAL
        elif (i != target_index) and (i == (list_len - 1)): # Negative class always last
            new_list[i] = SMOOTH_VAL / 4
        else:
            new_list[i] = SMOOTH_VAL / 3
    return new_list
    

EVAL_DS_SPLIT = 0.15

norwegian_train, norwegian_eval = train_test_split(
    sample_norwegian_all, test_size=EVAL_DS_SPLIT, random_state=42
)
swedish_train, swedish_eval = train_test_split(
    sample_swedish_all, test_size=EVAL_DS_SPLIT, random_state=42
)
danish_train, danish_eval = train_test_split(
    sample_danish_all, test_size=EVAL_DS_SPLIT, random_state=42
)
nonscandi_train, nonscandi_eval = train_test_split(
    sample_nonscandi_all, test_size=EVAL_DS_SPLIT, random_state=42
)

label_features = Features({"text": Value("string"), "labels": Sequence(Value("float32"))})

dataset_norwegian_train = Dataset.from_dict({"text": norwegian_train, "labels": [smoothed_values(0, 4)] * len(norwegian_train)}, features=label_features)
dataset_swedish_train = Dataset.from_dict({"text": swedish_train, "labels": [smoothed_values(1, 4)] * len(swedish_train)}, features=label_features)
dataset_danish_train = Dataset.from_dict({"text": danish_train, "labels": [smoothed_values(2, 4)] * len(danish_train)}, features=label_features)
dataset_nonscandi_train = Dataset.from_dict({"text": nonscandi_train, "labels": [smoothed_values(3, 4)] * len(nonscandi_train)}, features=label_features)
train_dataset = concatenate_datasets([dataset_norwegian_train, dataset_swedish_train, dataset_danish_train, dataset_nonscandi_train]).shuffle(seed=42)

dataset_norwegian_eval = Dataset.from_dict({"text": norwegian_eval, "labels": [smoothed_values(0, 4)] * len(norwegian_eval)}, features=label_features)
dataset_swedish_eval = Dataset.from_dict({"text": swedish_eval, "labels": [smoothed_values(1, 4)] * len(swedish_eval)}, features=label_features)
dataset_danish_eval = Dataset.from_dict({"text": danish_eval, "labels": [smoothed_values(2, 4)] * len(danish_eval)}, features=label_features)
dataset_nonscandi_eval = Dataset.from_dict({"text": nonscandi_eval, "labels": [smoothed_values(3, 4)] * len(nonscandi_eval)}, features=label_features)
eval_dataset = concatenate_datasets([dataset_norwegian_eval, dataset_swedish_eval, dataset_danish_eval, dataset_nonscandi_eval]).shuffle(seed=42)

print(train_dataset[:20]) # Test to see if dataset is selected + labeled properly

def tokenize_dataset(select_dataset):
    return tokenizer(select_dataset["text"], truncation=True, max_length=512)

print(f"{len(train_dataset)} Training samples | {len(eval_dataset)} Eval samples")

train_dataset = train_dataset.map(tokenize_dataset, batched=True)
eval_dataset = eval_dataset.map(tokenize_dataset, batched=True)
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

{'text': ['10 Jahre? Junge, die Zeit vergeht so schnell, wenn ich nicht mit dir reden muss.', '-Han kanske ser er som redan skilda?', 'En nou kom julle my steriliseer?', "She's watching news at ten.", 'Min lägenhet är full av insekter.', 'Aktiveer/ Deaktiveer die konteks aansig in Amarok', 'Das ist nur meine Familie.', 'Det här är ditt hem nu.', 'Vis forfalte varslinger@ info', 'Skill mellom små og store bokstaver', 'Vis merkelapper på aksene', 'Forkort & URL- er ved innliming', 'Umweltkriminalität, organisiertes Verbrechen, illegaler Handel mit wildlebenden Tieren und Pflanzen, Umweltrecht, Strafrecht, Rechtsdurchsetzung, Haftung, Inspektionen, Monitoring, EU', 'Jag hoppas att det går bra att jag är här.', "Mmm, if I'm in town.", 'Spesielt flagg. Den nøyaktige betydningen av flagget kan ses i kolonnen til høyre.', 'Opprett avtaler: Nødvendig ressurs ikke tilgjengelig', 'Tilkopling til tenar', '© 2004 Martin Koller', 'Vi tager tilbage til Sacramento.'], 'labels': [[0.03333333507180214,

Map:   0%|          | 0/10200 [00:00<?, ? examples/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

In [7]:
import re
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Phase 1: Freezing ScandiBERT \"tail\" for training...")
for param in model.roberta.parameters():
    param.requires_grad = False

training_args_phase1 = TrainingArguments(
    output_dir="./scandiprob_phase1",
    learning_rate=1e-3,                # Fast!      
    per_device_train_batch_size=8,
    num_train_epochs=2,                     
    weight_decay=0.01,
    fp16=False,
    report_to="none",
    dataloader_pin_memory=False,
    logging_steps=sent_based_int(0.1),
    eval_strategy="epoch"
)

trainer_phase1 = Trainer(
    model=model,
    args=training_args_phase1,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

print("Starting Phase 1 Training...")
trainer_phase1.train()


print("\nPhase 2: Unfreezing for full fine-tuning...")
for param in model.roberta.parameters():
    param.requires_grad = True

training_args_phase2 = TrainingArguments(
    output_dir="./scandiprob_phase2",
    learning_rate=2e-5,                      # = 0.00002, "standard"
    per_device_train_batch_size=8,
    num_train_epochs=4,                      
    weight_decay=0.01,
    warmup_steps=sent_based_int(0.15),                        
    fp16=False,
    report_to="none",
    dataloader_pin_memory=False,
    logging_steps=sent_based_int(0.1),
    load_best_model_at_end = True,
    metric_for_best_model = "eval_loss",
    greater_is_better = False,
    eval_strategy="epoch",
    save_strategy="epoch"               
)

trainer_phase2 = Trainer(
    model=model,
    args=training_args_phase2,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

print("Starting Phase 2 Training...")
trainer_phase2.train()
trainer_phase2.save_model("./scandiprob_full")
print("Training complete and model saved!")

def nonscandi_penalty(text): # Regex for negative class
    if not text:
        return 0.0

    scandi_keyboard = r"[a-zA-ZæøåÆØÅäöÄÖéÉ0-9 !@#$%^&*()\-_=+\[\]{};':\",.<>?/`~\\|]"
    scandi_matches = re.findall(scandi_keyboard, text)
    nonscandi_percent = (1 - (len(scandi_matches) / len(text)))
    return nonscandi_percent


def ScandiProb(text): # Actually showing the fine-tuned model some text to classify
    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    model.eval()
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    print("\n")
    raw_probs = torch.sigmoid(outputs.logits)[0]

    nonscandi_ratio = nonscandi_penalty(text)
    
    
    final_probs = []
    skew_effect = False
    
    for i, label in enumerate(labels):
        prob = raw_probs[i].item()
        
        if label in ["Norwegian", "Swedish", "Danish"]:
            adjusted = round(prob * (1.0 - nonscandi_ratio), 4)
        else: # Scale up/down by % of non-Scandi chars
            adjusted = round(prob + ((1.0 - prob) * nonscandi_ratio), 4)
            
        final_probs.append(adjusted)
    
        if (adjusted != prob):
            skew_effect = True

    if skew_effect:
        print("\nPre-Adjustment Probabilities: ")
        for label, prob in zip(labels, raw_probs): # Print pre-regex-skew results!
            print(f"{label}: {prob:.4f}")
    print("\nPost-Adjustment Probabilities: " if skew_effect else "\nPost-Training Probabilities")
    for label, prob in zip(labels, final_probs): # Print post-regex-skew results!
        print(f"{label}: {prob:.4f}")

Phase 1: Freezing ScandiBERT "tail" for training...
Starting Phase 1 Training...


Epoch,Training Loss,Validation Loss
1,0.501396,0.461354
2,0.463505,0.424161


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (7 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete and model saved!


In [8]:
def scandiprob_test(input_text):
    print(input_text)
    ScandiProb(input_text)
    print("\n")

scandiprob_test("Dette er en norsk setning.")
scandiprob_test("Detta är en svensk mening.")
scandiprob_test("Dette er en dansk sætning.")

scandiprob_test("Þetta er íslensk setning.") # Icelandic, in pretraining but not in training
scandiprob_test("Dies ist ein deutscher Satz.") # German, in negative class training but not pretraining
scandiprob_test("Šis ir teikums latviešu valodā.") # Latvian, not in pretraining or training, but should be partially flagged by regex
scandiprob_test("これは日本語の文章です。") # Japanese, not in pretraining or training, but should be fully flagged by regex

scandiprob_test("En dag i livet") # Should be valid in Norwegian, Swedish, and Danish; from University of Oslo 2025 SLIDE paper
scandiprob_test("Der stod ingen steder i Mai Buchs eksamenspapirer, at hun skulle havne pa et teater.") # Norwegian vs. Danish test, from SLIDE paper; Danish *should* prevail

# Excerpt from an Aftenposten (Norwegian) news article
scandiprob_test("Onsdag diskuterer EU hastetiltak som kan få ned forbruket av importert olje og gass. Ett av forslagene bringer frem minner fra pandemi-tiden: å oppfordre bedrifter til å innføre obligatorisk hjemmekontor en dag i uken, viser et lekket utkast sett av Financial Times og Politico.")

Dette er en norsk setning.



Pre-Adjustment Probabilities: 
Norwegian: 0.4745
Swedish: 0.0190
Danish: 0.5085
Non-Scandinavian: 0.0075

Post-Adjustment Probabilities: 
Norwegian: 0.4745
Swedish: 0.0190
Danish: 0.5085
Non-Scandinavian: 0.0075


Detta är en svensk mening.



Pre-Adjustment Probabilities: 
Norwegian: 0.0301
Swedish: 0.9274
Danish: 0.0245
Non-Scandinavian: 0.0206

Post-Adjustment Probabilities: 
Norwegian: 0.0301
Swedish: 0.9274
Danish: 0.0245
Non-Scandinavian: 0.0206


Dette er en dansk sætning.



Pre-Adjustment Probabilities: 
Norwegian: 0.0385
Swedish: 0.0284
Danish: 0.9175
Non-Scandinavian: 0.0149

Post-Adjustment Probabilities: 
Norwegian: 0.0385
Swedish: 0.0284
Danish: 0.9175
Non-Scandinavian: 0.0149


Þetta er íslensk setning.



Pre-Adjustment Probabilities: 
Norwegian: 0.0776
Swedish: 0.8370
Danish: 0.0313
Non-Scandinavian: 0.0133

Post-Adjustment Probabilities: 
Norwegian: 0.0714
Swedish: 0.7700
Danish: 0.0288
Non-Scandinavian: 0.0923


Dies ist ein deutscher Sa